In [177]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random

THRESHOLD_TIMESTAMPS = 16
L_SEQUENCE_LENGHT = 48

In [178]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

objects = extract_json("train.jsonl")

print(type(objects))


<class 'generator'>


In [179]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [180]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [181]:
sessions_in_dataset = OttoDataSetSession(sessions_bf_threshold)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght_after_threshold = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]["timestamps"]
    if len(sample) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(sample)

#print(np.mean(session_sample_lenght_after_threshold))
#print(f"The total of the median is for this total of {len(session_sample_lenght_after_threshold)} {np.median(session_sample_lenght_after_threshold)}")

Total len of the Sessions: 201


In [196]:
class CutoOttoDataSet(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        self.session = session
    
    def __cut_training_tesing__(self, min_value=0.70,max_value=0.80):
        training_batches = []
        testing_batches = []
        for single_session in self.session:       
            cutting_number = random.uniform(min_value,max_value)
            cut_index = int(len(single_session) * cutting_number )
            diff = np.absolute(cut_index - len(single_session))
            training_batches.append(cut_index)
            testing_batches.append(diff)
            
        return training_batches, testing_batches
    
    def __sequenceLenght__(self):
        return [L_SEQUENCE_LENGHT if i >= L_SEQUENCE_LENGHT else "Add_padding" for i in self.__cut_training_tesing__(0.70,0.80)[0]]
        

In [197]:
data_set_after_L = CutoOttoDataSet(session_sample_lenght_after_threshold)
training, testing = data_set_after_L.__cut_training_tesing__()
print(f"Training Batch: {len(training)}")
print(f"Testing Batch: {len(testing)}")

print(data_set_after_L.__sequenceLenght__())


Training Batch: 154
Testing Batch: 154
[48, 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 48, 48, 48, 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 48, 48, 'Add_padding', 'Add_padding', 48, 'Add_padding', 48, 'Add_padding', 'Add_padding', 'Add_padding', 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 'Add_padding', 'Add_padding', 48, 48, 48, 48, 'Add_padding', 48, 'Add_padding', 'Add_paddi